<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/01-glowtts-from-scratch/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Coqui TTS has dependency-injection style api for working with its function and classes.

I am using GlowTTS and LJSpeech for trainig single speaker text-to-speech model.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!nvidia-smi

Mon Aug  3 11:12:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install coqui-tts

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=f0ea502c69a69504640c441339dd6970617d0ebce7369bb4014f96831b130085
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [4]:
# Sanity check -- confirm CUDA + GPU are visible before going further
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), "CUDA not available -- check GPU accelerator is enabled in notebook settings"

torch: 2.10.0+cu128
CUDA available: True
GPU 0: Tesla T4


In [5]:
!pip uninstall -y transformers
!pip install transformers==4.57.5

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [6]:
from TTS.api import TTS

print("TTS OK")

TTS OK


In [7]:
# Kaggle's /dev/shm is small, which makes the default multiprocessing
# sharing strategy crash DataLoader workers with a bus error once
# num_workers > 0. Routing tensor-passing through disk instead fixes it.
import torch.multiprocessing as mp
mp.set_sharing_strategy("file_system")

In [8]:
# !pip freeze > requirements.txt

In [9]:

if not os.path.isdir("./data/LJSpeech-1.1/wavs"):
    from TTS.utils.downloaders import download_ljspeech
    dataset_path = download_ljspeech("./data")
else:
    dataset_path = "./data/LJSpeech-1.1"
print(dataset_path)

100%|██████████| 2.56G/2.56G [00:34<00:00, 78.6MB/s]


None


In [10]:
# verify dataset
import os
import soundfile as sf

path = "./data/LJSpeech-1.1/wavs"

files = os.listdir(path)

bad = 0

for f in files[:100]:
    try:
        audio, sr = sf.read(os.path.join(path, f))
        if sr != 22050:
            bad += 1
    except:
        bad += 1

print("bad:", bad)
print("files:", len(files))

bad: 0
files: 13100


In [11]:
# read transcripts
import pandas as pd

path = "./data/LJSpeech-1.1"
metadata = pd.read_csv(
    os.path.join(path, "metadata.csv"),
    sep="|",
    header=None,
)

metadata.head()

,0,1,2
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...


In [12]:
# create dataset config
from TTS.tts.configs.shared_configs import BaseDatasetConfig

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    path="./data/LJSpeech-1.1/",
    meta_file_train="metadata.csv"
)

In [13]:
# GlowTTS model config
from TTS.tts.configs.glow_tts_config import GlowTTSConfig

config = GlowTTSConfig(
    batch_size = 16,
    eval_batch_size = 8,
    num_loader_workers = 2,
    num_eval_loader_workers = 2,
    run_eval = True,
    epochs = 60,
    text_cleaner = "english_cleaners",
    use_phonemes = False,
    print_step = 25,
    mixed_precision = True,  # T4 has Tensor Cores -- AMP gives a real speedup here
    output_path = "./checkpoints",
    datasets = [dataset_config],
)

print("GlowTTS config ready")

GlowTTS config ready


In [14]:
# prepare data for model training flow
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.datasets import load_tts_samples

# convert wav files into mel-spectrograms
ap = AudioProcessor.init_from_config(config)

# convert text into token IDs
tokenizer, config = TTSTokenizer.init_from_config(config)

# load dataset
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=config.eval_split_size,
    eval_split_max_size=config.eval_split_max_size
)

print("Train: ", len(train_samples))
print("Eval: ", len(eval_samples))

Train:  12969
Eval:  131


In [15]:
# GlowTTS model initialization
from TTS.tts.models.glow_tts import GlowTTS

model = GlowTTS(
    config,
    ap,
    tokenizer,
    speaker_manager=None
)

print("GlowTTS model ready")

GlowTTS model ready


In [16]:
# trainer setup
from trainer import Trainer, TrainerArgs

trainer = Trainer(
    TrainerArgs(),
    config,
    config.output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples
)

print("Trainer ready")

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 4
 | > Num. of Torch Threads: 2
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=checkpoints/run-August-03-2026_11+18AM-0000000

 > Model has 28597969 parameters


Trainer ready


In [17]:
# training
trainer.fit()


 > EPOCH: 0/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 11:20:39) 

   --> TIME: 2026-08-03 11:20:43 -- STEP: 0/810 -- GLOBAL_STEP: 0
     | > current_lr: 2.5e-07  (2.5e-07)
     | > step_time: 3.4109  (3.4108664989471436)
     | > loader_time: 0.7867  (0.7866942882537842)

 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.

   --> 


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.007191121578216552 (-0.0012282580137252816)
     | > avg_loss: -0.025556446984410286 (-0.008649068884551525)
     | > avg_log_mle: -0.2605053633451461 (-0.004666022956371252)
     | > avg_loss_dur: 0.2349489163607359 (-0.003983045928180218)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_30007.pth

 > EPOCH: 37/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 14:49:41) 

   --> TIME: 2026-08-03 14:49:48 -- STEP: 18/810 -- GLOBAL_STEP: 30025
     | > current_lr: 9.5e-06  (9.500000000000006e-06)
     | > loss: 0.03511759638786316  (0.005741388433509403)
     | > log_mle: -0.23524534702301025  (-0.22240716881222194)
     | > loss_dur: 0.2703629434108734  (0.22814855724573135)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(5.6145, device='cuda:0')  (tensor(5.5512, device='cuda:0'))
     | > step_time: 0.3299  (0.316881431473626)
     | > loader_time: 0.003  (0.003


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.00845654308795929 (+0.0001415163278579712)
     | > avg_loss: -0.036942124366760254 (-0.01218516007065773)
     | > avg_log_mle: -0.2698880210518837 (-0.011599250137805939)
     | > avg_loss_dur: 0.23294589668512344 (-0.0005859099328517914)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_31629.pth

 > EPOCH: 39/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 15:01:01) 

   --> TIME: 2026-08-03 15:01:08 -- STEP: 21/810 -- GLOBAL_STEP: 31650
     | > current_lr: 1e-05  (1e-05)
     | > loss: -0.03879298269748688  (-0.013350110678445725)
     | > log_mle: -0.24086904525756836  (-0.23062098026275635)
     | > loss_dur: 0.20207606256008148  (0.21727086958431063)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(5.6471, device='cuda:0')  (tensor(7.4243, device='cuda:0'))
     | > step_time: 0.3287  (0.31499695777893066)
     | > loader_time: 0.0033  (0.003556206112816


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.008749350905418396 (+0.00029280781745910645)
     | > avg_loss: -0.04302947875112295 (-0.006087354384362698)
     | > avg_log_mle: -0.272444024682045 (-0.0025560036301612854)
     | > avg_loss_dur: 0.22941454593092203 (-0.003531350754201412)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_32440.pth

 > EPOCH: 40/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 15:06:42) 

   --> TIME: 2026-08-03 15:06:47 -- STEP: 10/810 -- GLOBAL_STEP: 32450
     | > current_lr: 1.025e-05  (1.0250000000000002e-05)
     | > loss: 0.016848981380462646  (-0.011657103896141052)
     | > log_mle: -0.2248067855834961  (-0.23402384519577027)
     | > loss_dur: 0.24165576696395874  (0.22236674129962922)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(2.6405, device='cuda:0')  (tensor(7.7656, device='cuda:0'))
     | > step_time: 0.3181  (0.32261269092559813)
     | > loader_time: 0.003


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.007399559020996093 (-0.0013497918844223031)
     | > avg_loss: -0.04461376368999481 (-0.0015842849388718605)
     | > avg_log_mle: -0.27022457122802734 (+0.002219453454017639)
     | > avg_loss_dur: 0.22561080753803253 (-0.0038037383928894997)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_33251.pth

 > EPOCH: 41/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 15:12:23) 

   --> TIME: 2026-08-03 15:12:32 -- STEP: 24/810 -- GLOBAL_STEP: 33275
     | > current_lr: 1.0500000000000001e-05  (1.0500000000000006e-05)
     | > loss: -0.025082141160964966  (-0.021851820250352223)
     | > log_mle: -0.2581678628921509  (-0.2392598936955134)
     | > loss_dur: 0.2330857217311859  (0.2174080734451612)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(4.8276, device='cuda:0')  (tensor(6.1181, device='cuda:0'))
     | > step_time: 0.3471  (0.31717212994893385)
     | > loade

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.006409347057342529 (-0.0005640685558319092)
     | > avg_loss: -0.12164057604968548 (-0.003179808147251606)
     | > avg_log_mle: -0.32519038021564484 (-0.001852273941040039)
     | > avg_loss_dur: 0.20354980416595936 (-0.001327534206211567)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_45416.pth

 > EPOCH: 56/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 16:37:59) 

   --> TIME: 2026-08-03 16:38:03 -- STEP: 9/810 -- GLOBAL_STEP: 45425
     | > current_lr: 1.425e-05  (1.4249999999999997e-05)
     | > loss: -0.0891425758600235  (-0.10949478712346819)
     | > log_mle: -0.276919960975647  (-0.2910004324383206)
     | > loss_dur: 0.18777738511562347  (0.1815056453148524)
     | > amp_scaler: 2048.0  (2048.0)
     | > grad_norm: tensor(4.2971, device='cuda:0')  (tensor(14.4573, device='cuda:0'))
     | > step_time: 0.2747  (0.29810484250386554)
     | > loader_time: 0.0031  (0


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.00866726040840149 (+0.00225791335105896)
     | > avg_loss: -0.12673954851925373 (-0.0050989724695682526)
     | > avg_log_mle: -0.32876610755920416 (-0.0035757273435593206)
     | > avg_loss_dur: 0.20202655903995037 (-0.0015232451260089874)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_46227.pth

 > EPOCH: 57/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 16:43:40) 

   --> TIME: 2026-08-03 16:43:49 -- STEP: 23/810 -- GLOBAL_STEP: 46250
     | > current_lr: 1.4500000000000002e-05  (1.4500000000000009e-05)
     | > loss: -0.11635337769985199  (-0.10920321811800418)
     | > log_mle: -0.29563355445861816  (-0.29253541904947034)
     | > loss_dur: 0.17928017675876617  (0.18333220093146615)
     | > amp_scaler: 2048.0  (2048.0)
     | > grad_norm: tensor(11.8004, device='cuda:0')  (tensor(18.0609, device='cuda:0'))
     | > step_time: 0.346  (0.3262294997339663)
     | > loade


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.008146822452545166 (-0.0005204379558563232)
     | > avg_loss: -0.13119044248014688 (-0.004450893960893154)
     | > avg_log_mle: -0.3322416692972183 (-0.0034755617380141657)
     | > avg_loss_dur: 0.20105122681707144 (-0.000975332222878933)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_47038.pth

 > EPOCH: 58/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 16:49:19) 

   --> TIME: 2026-08-03 16:49:24 -- STEP: 12/810 -- GLOBAL_STEP: 47050
     | > current_lr: 1.475e-05  (1.4750000000000004e-05)
     | > loss: -0.12406380474567413  (-0.11242471386988957)
     | > log_mle: -0.2906001806259155  (-0.29306747515996295)
     | > loss_dur: 0.1665363758802414  (0.1806427612900734)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(9.6125, device='cuda:0')  (tensor(18.9753, device='cuda:0'))
     | > step_time: 0.2851  (0.3092004060745239)
     | > loader_time: 0.0035  


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.007420554757118223 (-0.0007262676954269427)
     | > avg_loss: -0.1349476221948862 (-0.0037571797147393227)
     | > avg_log_mle: -0.3346416652202606 (-0.0023999959230422974)
     | > avg_loss_dur: 0.1996940430253744 (-0.0013571837916970253)

 > BEST MODEL : checkpoints/run-August-03-2026_11+18AM-0000000/best_model_47849.pth

 > EPOCH: 59/59
 --> checkpoints/run-August-03-2026_11+18AM-0000000

 > TRAINING (2026-08-03 16:54:58) 

   --> TIME: 2026-08-03 16:55:00 -- STEP: 1/810 -- GLOBAL_STEP: 47850
     | > current_lr: 1.5e-05  (1.5e-05)
     | > loss: -0.11001093685626984  (-0.11001093685626984)
     | > log_mle: -0.2864793539047241  (-0.2864793539047241)
     | > loss_dur: 0.17646841704845428  (0.17646841704845428)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(11.2806, device='cuda:0')  (tensor(11.2806, device='cuda:0'))
     | > step_time: 0.2951  (0.29506564140319824)
     | > loader_time: 0.0058  (0.005764245986


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.00843106210231781 (+0.0010105073451995867)
     | > avg_loss: -0.12674127984791994 (+0.008206342346966267)
     | > avg_log_mle: -0.3267877325415611 (+0.007853932678699493)
     | > avg_loss_dur: 0.20004645269364119 (+0.0003524096682667732)



In [18]:
# tensorboard monitoring
%load_ext tensorboard

%tensorboard --logdir ./checkpoints

<IPython.core.display.Javascript object>